In [1]:
from wien2k import *

"""
MnCrAs doubled
1.0 Ang
3.5779930000000002 0.0000000000000002 0.0000000000000002
0.0000000000000000 3.5779930000000002 0.0000000000000002
0.0000000000000000 0.0000000000000000 12.2525300000000001
Cr Mn As 
4 4 4 
Direct
0.7500000000000000 0.7500000000000000 0.3326450000000000 Cr
0.2500000000000000 0.2500000000000000 0.1673550000000000 Cr
0.7500000000000000 0.7500000000000000 0.8326450000000001 Cr
0.2500000000000000 0.2500000000000000 0.6673550000000000 Cr
0.7500000000000000 0.2500000000000000 0.0000000000000000 Mn
0.2500000000000000 0.7500000000000000 0.0000000000000000 Mn
0.7500000000000000 0.2500000000000000 0.5000000000000000 Mn
0.2500000000000000 0.7500000000000000 0.5000000000000000 Mn
0.7500000000000000 0.7500000000000000 0.1359680000000000 As
0.2500000000000000 0.2500000000000000 0.3640320000000000 As
0.7500000000000000 0.7500000000000000 0.6359680000000000 As
0.2500000000000000 0.2500000000000000 0.8640320000000000 As
"""

mncras = """
Mn2 Cr2 As2
1.0 Ang
   3.5779930000000002    0.0000000000000000    0.0000000000000002
   0.0000000000000006    3.5779930000000002    0.0000000000000002
   0.0000000000000000    0.0000000000000000    6.1262650000000001
Mn Cr As
2 2 2
direct
   0.0000000000000000    0.0000000000000000    0.0000000000000000 Mn
   0.5000000000000000    0.5000000000000000    0.0000000000000000 Mn
   0.0000000000000000    0.5000000000000000    0.6652900000000000 Cr
   0.5000000000000000    0.0000000000000000    0.3347100000000000 Cr
   0.0000000000000000    0.5000000000000000    0.2719360000000001 As
   0.5000000000000000    0.0000000000000000    0.7280639999999998 As

"""

# load the structure from materials project
# struct = StructureFile.load_materials_project(
#     "https://next-gen.materialsproject.org/materials/mp-1221644?formula=CrMnAs",  # load in MnCrAs
#     "credentials.json",
# )
struct = StructureFile.parse_poscar(mncras)
struct.tweak_cell_multiples(c=2)
orig_poscar = struct.generate_poscar()

print("Original POSCAR loaded")
print(orig_poscar)

Original POSCAR loaded
Mn2 Cr2 As2
1.0 Ang
3.5779930000000002 0.0000000000000002 0.0000000000000002
0.0000000000000000 3.5779930000000002 0.0000000000000002
0.0000000000000000 0.0000000000000000 12.2525300000000001
Cr Mn As 
4 4 4 
Direct
0.0000000000000000 0.5000000000000000 0.3326450000000000 Cr
0.5000000000000000 0.0000000000000000 0.1673550000000000 Cr
0.0000000000000000 0.5000000000000000 0.8326450000000001 Cr
0.5000000000000000 0.0000000000000000 0.6673550000000000 Cr
0.0000000000000000 0.0000000000000000 0.0000000000000000 Mn
0.5000000000000000 0.5000000000000000 0.0000000000000000 Mn
0.0000000000000000 0.0000000000000000 0.5000000000000000 Mn
0.5000000000000000 0.5000000000000000 0.5000000000000000 Mn
0.0000000000000000 0.5000000000000000 0.1359680000000001 As
0.5000000000000000 0.0000000000000000 0.3640319999999999 As
0.0000000000000000 0.5000000000000000 0.6359680000000001 As
0.5000000000000000 0.0000000000000000 0.8640319999999999 As



In [2]:
# find valid permutations (8 choose 4) for CrMnAs case
valid_combinations = []
for i in range(2**8):
    bin_rep = f"{i:08b}"

    if bin_rep.count("0") == 4:
        # passes combination check
        valid_combinations.append(bin_rep)

print("Valid combinations found", len(valid_combinations))

Valid combinations found 70


In [3]:
# generate all the valid combination structures
combination_structures = []
for comb in valid_combinations:
    struct_copy = StructureFile.parse_poscar(orig_poscar)
    struct_copy.title = f"comb_{comb}"

    for i in range(8):
        struct_copy.tweak_atom(i, Z=(24 if comb[i] == "0" else 25))  # Cr or Mn

    combination_structures.append(struct_copy)

print("Combination structures generated", len(combination_structures))

Combination structures generated 70


In [4]:
equivalence_matrix = np.zeros(
    (len(combination_structures), len(combination_structures))
).astype(int)

if not os.path.exists("_equivalence_matrix.txt"):

    for i in range(len(combination_structures)):

        s1 = combination_structures[i]


        for j in range(len(combination_structures)):

            s2 = combination_structures[j]


            # print(i, j)

            # if s1 != s2:

            #     are_equiv, proof = s1.translational_equivalence_check(s2)

            #     print(are_equiv, proof)


            #     if are_equiv:

            #         equivalence_matrix[i][j] = 1

            # else:

            #     equivalence_matrix[i][j] = 1


            are_equiv, proof = s1.translational_equivalence_check(s2)


            if are_equiv:
                # print(valid_combinations[i], valid_combinations[j], proof)
                # print("\n".join([str(a) for a in s1.atoms]))
                # print("---------")
                # print("\n".join([str(a) for a in s2.atoms]))
                # print("\n ============= \n")

                equivalence_matrix[i][j] = 1


    with open("_equivalence_matrix.txt", "w+") as writer:

        writer.write(

            "\n".join(

                [
                    "".join(["{:2}".format(item).strip() for item in row])
                    for row in equivalence_matrix
                ]
            )

        )

    print("Equivalence matrix created")
else:
    with open("_equivalence_matrix.txt", "r+") as reader:
        equivalence_matrix = [[int(n) for n in row.split(" ")] for row in reader.readlines()]

    print("Equivalence matrix loaded")

Equivalence matrix loaded


In [ ]:
rows_to_check = list(range(len(combination_structures)))

necessary_combinations = []

unique_structures = []
necessary_structures = []
while len(rows_to_check) > 0:
    r = rows_to_check[0]
    
    necessary_structures.append(combination_structures[r])
    necessary_combinations.append(valid_combinations[r])
    # print(valid_combinations[r])
    
    count = 0
    for c in range(len(equivalence_matrix[r])):
        if equivalence_matrix[r][c] == 1:
            rows_to_check.remove(c)
            count+=1 
            
    if count == 1:
        unique_structures.append(combination_structures[r])

# # swap bottom cell with top cell => none found
# for c1 in [str(c) for c in necessary_combinations]:
#     for c2 in [str(c) for c in necessary_combinations]:
#         # print(c1, c2)
#         if(
#             c1[0] == c2[2] and
#             c1[1] == c2[3] and
#             c1[4] == c2[6] and
#             c1[5] == c2[7] and
#             #
#             c1[2] == c2[0] and
#             c1[3] == c2[1] and
#             c1[6] == c2[4] and
#             c1[7] == c2[5] and
#             #
#             c1 != c2
#            ):
#             print(f"{c1} == {c2}")

print("Unique structures found", len(unique_structures), "from", len(combination_structures))
print("Necessary structures found", len(necessary_structures), "from", len(combination_structures))

Unique structures found 6 from 70
Necessary structures found 38 from 70


: 

In [ ]:
mf = MaterialFolder("credentials.json", "CrMnAs", structure=mncras)
mf.open()
mf.cmd.type("mkdir combinations")
mf.cmd.cd("combinations")

lstart_pattern = ['u', 'd', 'd', 'u', 'u', 'u', 'd', 'd', 'n', 'n', 'n', 'n'] # AF3

for s in necessary_structures[0:4]:
    # set to the other structure
    mf.structure = s
    mf.manual_run(
        f"dbl_{s.title}",
        init_lapw_Parameters(
            kpoints=250,
            spin_polarized=True,
            lstart_flag="ask",
            x_ask_flags_pattern=s.generate_poscar_corresponding_lstart_pattern(lstart_pattern),
        ),
        auto_confirm=True,
    )

If the calculation finishes and the rest of the program is not able to recognise that, type 'manual_stop' into the ssh console to stop it manually and wait for the next check.


Socket exception: An existing connection was forcibly closed by the remote host (10054)
